In [1]:
# model_xlnet_bigru.py
# ─────────────────────────────────────────────────────────────────────────────
# XLNet + Signal-Cross-Attention + MHA + BiGRU
# Full 5000 docs · 50 epochs · Resume-safe · Full metrics + adaptive HP
#
# ══ NOVEL ADDITIONS ══════════════════════════════════════════════════════════
#  1. SIGNAL CROSS-ATTENTION
#  2. DEFERRED CLASS REWEIGHTING
#  3. LAYER-WISE LR DECAY (LLRD)
#  4. DYNAMIC CHUNK DROPOUT
#  5. STOCHASTIC WEIGHT AVERAGING (SWA)
#  6. ADAPTIVE HYPERPARAMETER ADJUSTMENT (per-epoch)
#  7. FULL EVALUATION METRICS every epoch
#  8. RESUME-SAFE CHECKPOINT
#
# ══ ARCHITECTURE ═════════════════════════════════════════════════════════════
#  Backbone : XLNet (xlnet-base-cased)
#  Sequence : BiGRU
#
# ══ XLNet INTERNAL LAYOUT (XLNetModel when loaded via AutoModel) ═════════════
#  • self.word_embedding   – nn.Embedding  (token embeddings)
#  • self.mask_emb         – nn.Parameter  (NOT a sub-module; handle separately)
#  • self.layer            – nn.ModuleList of XLNetLayer  (12 for base)
#  • NO .transformer / .encoder attributes exist at this level
#  • No dedicated [CLS]; last token of last_hidden_state used as chunk repr.
#
# ══ BUG FIX (retained) ═══════════════════════════════════════════════════════
#  All XLNet encoder layers pre-registered in optimizer → scheduler group
#  count never changes → no runtime add_param_group() needed.
# ─────────────────────────────────────────────────────────────────────────────

import os, gc, json, random, logging, warnings, csv, math
from copy import deepcopy
from datetime import datetime
from collections import defaultdict, Counter
from typing import Optional

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
INPUT_PATH = "Track_C_balanced.jsonl"
OUTPUT_DIR = "single_run_results_TrackC_XLNet"
LOG_DIR    = f"{OUTPUT_DIR}/logs"
PLOT_DIR   = f"{OUTPUT_DIR}/plots"
CKPT_ROLL  = f"{OUTPUT_DIR}/checkpoint_last.pt"
CKPT_BEST  = f"{OUTPUT_DIR}/best_model.pt"
CSV_PATH   = f"{OUTPUT_DIR}/epoch_results.csv"

XLNET_MODEL_ID = "xlnet/xlnet-base-cased"

MAX_TOTAL_DOCS = 5000
MAX_EPOCHS     = 50
EARLY_STOP_PAT = 15
BATCH_SIZE     = 8
ACCUM_STEPS    = 2         # effective batch = 16

LR_XLNET       = 2e-5
LR_HEAD        = 1e-5
LLRD_DECAY     = 0.95
WARMUP_RATIO   = 0.06
WEIGHT_DECAY   = 0.01

MAX_CHUNK_LEN       = 256
MAX_CHUNKS          = 4
GRU_HIDDEN          = 256
GRU_LAYERS          = 2
GRU_DROPOUT         = 0.1
MHA_HEADS           = 8
MHA_DROPOUT         = 0.1
DROPOUT             = 0.1
FREEZE_XLNET_LAYERS = 6

WITH_SIGNAL         = True
LABEL_SMOOTHING     = 0.05
DEFERRED_RW_EPOCH   = 6
CHUNK_DROP_PROB     = 0.15
SWA_START           = 35
SWA_LR              = 5e-6

OVERFIT_GAP_THRESH  = 0.15
OVERFIT_PATIENCE    = 3
UNDERFIT_F1_THRESH  = 0.55

SEED     = 42
SOTA_F1  = 0.8131
SOTA_ACC = 0.78
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP  = DEVICE == "cuda"

for d in [OUTPUT_DIR, LOG_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING
# ══════════════════════════════════════════════════════════════════════════════
run_id   = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"{LOG_DIR}/run_{run_id}.log"
logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s | %(message)s",
    datefmt  = "%H:%M:%S",
    handlers = [logging.FileHandler(log_file), logging.StreamHandler()],
)
log = logging.getLogger()
log.info(f"Device        : {DEVICE}  |  AMP: {USE_AMP}")
log.info(f"Architecture  : XLNet → SignalCrossAttn → MHA({MHA_HEADS}h) → BiGRU → AttnPool → Linear")
log.info(f"Epochs        : {MAX_EPOCHS}  patience={EARLY_STOP_PAT}  SWA from ep {SWA_START}")
log.info(f"LR XLNET/HEAD : {LR_XLNET}/{LR_HEAD}  LLRD={LLRD_DECAY}  WD={WEIGHT_DECAY}")
log.info(f"Novel         : SignalCrossAttn + DeferredRW(ep{DEFERRED_RW_EPOCH}) + LLRD + ChunkDrop + SWA + AdaptiveHP")
log.info(f"Fix           : All XLNet layers pre-registered in optimizer — no runtime add_param_group")


# ══════════════════════════════════════════════════════════════════════════════
# SEED
# ══════════════════════════════════════════════════════════════════════════════
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(SEED)
torch.backends.cudnn.enabled   = True
torch.backends.cudnn.benchmark = True


# ══════════════════════════════════════════════════════════════════════════════
# SIGNAL TOKENS
# ══════════════════════════════════════════════════════════════════════════════
SIGNAL_MAP = {
    "FAVORS_PETITIONER": "[FP]",
    "FAVORS_RESPONDENT": "[FR]",
    "NEUTRAL"          : "[N]",
}
SIGNAL_IDX = {"FAVORS_PETITIONER": 0, "FAVORS_RESPONDENT": 1, "NEUTRAL": 2}

def format_input(question, answer, signal, with_signal=True):
    sig = SIGNAL_MAP.get(signal, "[N]") if with_signal else ""
    return f"Q: {question.strip()} A: {answer.strip()} {sig}".strip()


# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════
class HierarchicalXLNetBiGRU(nn.Module):
    """
    XLNet + Signal-Cross-Attention + MHA + BiGRU + Attn-Pool

    XLNetModel (loaded via AutoModel) internal layout — verified from source:
      • xlnet.word_embedding   → nn.Embedding
      • xlnet.mask_emb         → nn.Parameter  (NOT a module)
      • xlnet.layer[i]         → nn.ModuleList (NOT .transformer.layer)
    No .transformer or .encoder attribute exists at the XLNetModel level.
    Pooling: no [CLS] → use last_hidden_state[:, -1, :] per chunk.
    """

    def __init__(self, model_id, num_labels=2, dropout=0.1,
                 gru_hidden=256, gru_layers=2, gru_dropout=0.1,
                 mha_heads=8, mha_dropout=0.1,
                 label_smoothing=0.05, freeze_xlnet_layers=6):
        super().__init__()
        self.label_smoothing     = label_smoothing
        self.freeze_xlnet_layers = freeze_xlnet_layers

        # ── XLNet backbone ────────────────────────────────────────────────────
        self.xlnet = AutoModel.from_pretrained(model_id)
        D = self.xlnet.config.d_model          # 768 for xlnet-base-cased
        self._freeze_xlnet(freeze_xlnet_layers)

        # ── Signal embedding ─────────────────────────────────────────────────
        self.signal_emb = nn.Embedding(3, D)
        nn.init.normal_(self.signal_emb.weight, std=0.02)

        # ── Signal cross-attention ───────────────────────────────────────────
        self.signal_cross_attn = nn.MultiheadAttention(
            embed_dim=D, num_heads=mha_heads,
            dropout=mha_dropout, batch_first=True,
        )
        self.signal_norm = nn.LayerNorm(D)

        # ── Chunk-level MHA ───────────────────────────────────────────────────
        self.chunk_mha   = nn.MultiheadAttention(
            embed_dim=D, num_heads=mha_heads,
            dropout=mha_dropout, batch_first=True,
        )
        self.mha_norm    = nn.LayerNorm(D)
        self.mha_dropout = nn.Dropout(mha_dropout)

        # ── BiGRU ─────────────────────────────────────────────────────────────
        bigru_out = gru_hidden * 2
        self.bigru = nn.GRU(
            input_size=D,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=gru_dropout if gru_layers > 1 else 0.0,
        )

        # ── Attention pooling + classifier ────────────────────────────────────
        self.attn_layer = nn.Linear(bigru_out, 1)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(bigru_out, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    # ── Freeze helpers ────────────────────────────────────────────────────────
    def _freeze_xlnet(self, n_layers):
        """
        Correct XLNetModel paths (verified from HuggingFace source line 866-868):
          self.xlnet.word_embedding   – nn.Embedding
          self.xlnet.layer[i]         – XLNetLayer blocks (NOT .transformer.layer)
        """
        for p in self.xlnet.word_embedding.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.xlnet.layer):      # ← .layer, not .transformer.layer
            for p in layer.parameters():
                p.requires_grad = (i >= n_layers)

    def unfreeze_xlnet_from(self, n_layers):
        self._freeze_xlnet(n_layers)
        self.freeze_xlnet_layers = n_layers
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        log.info(f"  [AdaptiveHP] XLNet unfrozen from layer {n_layers} "
                 f"→ {trainable:,} trainable params")

    # ── Chunk encoder ─────────────────────────────────────────────────────────
    def encode_chunks(self, chunk_input_ids, chunk_attention_mask, chunk_mask):
        B, N, L   = chunk_input_ids.shape
        flat_ids  = chunk_input_ids.view(B * N, L)
        flat_mask = chunk_attention_mask.view(B * N, L)
        out = self.xlnet(input_ids=flat_ids, attention_mask=flat_mask)
        # No [CLS] in XLNet — use last token as chunk summary
        cls = out.last_hidden_state[:, -1, :].view(B, N, -1)
        cls = cls * chunk_mask.unsqueeze(-1).float()
        return cls

    # ── Forward ───────────────────────────────────────────────────────────────
    def forward(self, chunk_input_ids, chunk_attention_mask, chunk_mask,
                signal_ids, labels=None, chunk_drop_prob=0.0):

        chunk_cls = self.encode_chunks(
            chunk_input_ids, chunk_attention_mask, chunk_mask)

        # Dynamic chunk dropout
        if chunk_drop_prob > 0.0 and self.training:
            drop_mask  = (torch.rand(chunk_cls.shape[:2],
                                     device=chunk_cls.device) > chunk_drop_prob)
            safe_mask  = chunk_mask.bool() & drop_mask
            any_real   = safe_mask.any(dim=1, keepdim=True)
            final_mask = torch.where(any_real, safe_mask, chunk_mask.bool())
            chunk_cls  = chunk_cls * final_mask.unsqueeze(-1).float()

        # Signal cross-attention
        sig_q      = self.signal_emb(signal_ids).unsqueeze(1)
        key_pad    = (chunk_mask == 0)
        sig_ctx, _ = self.signal_cross_attn(
            query=sig_q, key=chunk_cls, value=chunk_cls,
            key_padding_mask=key_pad,
        )
        sig_ctx   = self.signal_norm(sig_q + sig_ctx)
        chunk_ctx = chunk_cls + sig_ctx
        chunk_ctx = chunk_ctx * chunk_mask.unsqueeze(-1).float()

        # Chunk MHA
        mha_out, _ = self.chunk_mha(
            query=chunk_ctx, key=chunk_ctx, value=chunk_ctx,
            key_padding_mask=key_pad,
        )
        chunk_ctx = self.mha_norm(chunk_ctx + self.mha_dropout(mha_out))
        chunk_ctx = chunk_ctx * chunk_mask.unsqueeze(-1).float()

        # BiGRU — (output, h_n), no cell state
        gru_out, _ = self.bigru(chunk_ctx)

        # Attention pooling
        scores   = self.attn_layer(gru_out).squeeze(-1)
        scores   = scores.masked_fill(~chunk_mask.bool(), float("-inf"))
        weights  = F.softmax(scores, dim=1)
        doc_repr = (gru_out * weights.unsqueeze(-1)).sum(dim=1)

        logits = self.classifier(self.dropout(doc_repr))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels,
                                   label_smoothing=self.label_smoothing)

        class Out: pass
        o = Out(); o.loss = loss; o.logits = logits
        return o

    def resize_token_embeddings(self, n):
        self.xlnet.resize_token_embeddings(n)


# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════
class HierarchicalLegalQADataset(Dataset):
    def __init__(self, records, tokenizer,
                 max_chunk_len=256, max_chunks=4, with_signal=True):
        doc_groups = defaultdict(list)
        for r in records:
            doc_groups[r["doc_id"]].append(r)

        pad_ids  = torch.zeros(max_chunk_len, dtype=torch.long)
        pad_mask = torch.zeros(max_chunk_len, dtype=torch.long)

        self.samples = []
        for doc_id, qa_list in tqdm(doc_groups.items(),
                                    desc="  tokenising", leave=False):
            label    = int(qa_list[0]["label"])
            signals  = [r.get("signal", "NEUTRAL") for r in qa_list]
            dom_sig  = Counter(signals).most_common(1)[0][0]
            sig_idx  = SIGNAL_IDX.get(dom_sig, 2)

            texts   = [format_input(r["question"], r["answer"],
                                    r["signal"], with_signal)
                       for r in qa_list][:max_chunks]
            n_real  = len(texts)

            all_ids, all_mask = [], []
            for text in texts:
                enc = tokenizer(text, max_length=max_chunk_len,
                                padding="max_length", truncation=True,
                                return_tensors="pt")
                all_ids.append(enc["input_ids"].squeeze(0))
                all_mask.append(enc["attention_mask"].squeeze(0))

            while len(all_ids) < max_chunks:
                all_ids.append(pad_ids.clone())
                all_mask.append(pad_mask.clone())

            self.samples.append({
                "chunk_input_ids"     : torch.stack(all_ids),
                "chunk_attention_mask": torch.stack(all_mask),
                "chunk_mask"          : torch.tensor(
                    [1]*n_real + [0]*(max_chunks - n_real), dtype=torch.long),
                "label"               : torch.tensor(label, dtype=torch.long),
                "signal_id"           : torch.tensor(sig_idx, dtype=torch.long),
                "doc_id"              : doc_id,
            })

        log.info(f"  Dataset ready : {len(self.samples)} docs (pre-tokenised)")

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    return {
        "chunk_input_ids"     : torch.stack([b["chunk_input_ids"]        for b in batch]),
        "chunk_attention_mask": torch.stack([b["chunk_attention_mask"]    for b in batch]),
        "chunk_mask"          : torch.stack([b["chunk_mask"]              for b in batch]),
        "label"               : torch.stack([b["label"]                   for b in batch]),
        "signal_id"           : torch.stack([b["signal_id"]               for b in batch]),
        "doc_id"              : [b["doc_id"] for b in batch],
    }


# ══════════════════════════════════════════════════════════════════════════════
# DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def load_data(path):
    recs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip(): recs.append(json.loads(line))
    return recs


def build_balanced_pool(records, max_docs=5000, seed=42):
    random.seed(seed)
    dg = defaultdict(list)
    for r in records: dg[r["doc_id"]].append(r)
    ids = list(dg.keys()); random.shuffle(ids)

    def dlabel(d):
        l = [int(r["label"]) for r in dg[d]]
        return 1 if l.count(1) >= l.count(0) else 0

    c0 = [d for d in ids if dlabel(d) == 0]
    c1 = [d for d in ids if dlabel(d) == 1]
    n  = min(len(c0), len(c1), max_docs // 2)
    bal = set(c0[:n] + c1[:n])
    pr = [r for r in records if r["doc_id"] in bal]
    pi = [d for d in ids     if d           in bal]
    log.info(f"  Balanced pool : {len(bal):,} docs  ({n} per class)  QA={len(pr):,}")
    return pr, pi


def split_train_val(pool_records, pool_ids, seed=42):
    n      = len(pool_ids)
    n_val  = max(1, int(round(n * 0.20)))
    n_tr   = n - n_val
    tr_ids = set(pool_ids[:n_tr]); va_ids = set(pool_ids[n_tr:])
    tr = [r for r in pool_records if r["doc_id"] in tr_ids]
    va = [r for r in pool_records if r["doc_id"] in va_ids]
    log.info(f"  Train : {n_tr} docs ({len(tr):,} QA)  |  Val : {n_val} docs ({len(va):,} QA)")
    return tr, va, n_tr, n_val


# ══════════════════════════════════════════════════════════════════════════════
# LLRD OPTIMISER — all XLNet layers pre-registered
# ══════════════════════════════════════════════════════════════════════════════
def build_llrd_optimizer(model, lr_xlnet, lr_head, decay, weight_decay):
    """
    Correct XLNetModel attribute paths (verified from source):
      model.xlnet.word_embedding        – nn.Embedding
      model.xlnet.mask_emb              – nn.Parameter (NOT a sub-module)
      model.xlnet.layer[i]              – XLNetLayer (NOT .transformer.layer)
    """
    num_layers   = len(model.xlnet.layer)   # 12 for xlnet-base-cased
    param_groups = []

    # Word embedding
    emb_lr     = lr_xlnet * (decay ** num_layers)
    emb_params = list(model.xlnet.word_embedding.parameters())
    if emb_params:
        param_groups.append({
            "params"      : emb_params,
            "lr"          : emb_lr,
            "weight_decay": weight_decay,
            "name"        : "xlnet_word_emb",
        })

    # mask_emb is an nn.Parameter (not a module) — wrap directly
    if hasattr(model.xlnet, "mask_emb") and model.xlnet.mask_emb is not None:
        param_groups.append({
            "params"      : [model.xlnet.mask_emb],
            "lr"          : emb_lr,
            "weight_decay": 0.0,
            "name"        : "xlnet_mask_emb",
        })

    # ALL transformer blocks via .layer (frozen layers included)
    for i, layer in enumerate(model.xlnet.layer):
        layer_lr     = lr_xlnet * (decay ** (num_layers - i))
        layer_params = list(layer.parameters())
        if layer_params:
            param_groups.append({
                "params"      : layer_params,
                "lr"          : layer_lr,
                "weight_decay": weight_decay,
                "name"        : f"xlnet_layer_{i:02d}",
            })

    # Head
    head_params = (
        list(model.signal_emb.parameters())
        + list(model.signal_cross_attn.parameters())
        + list(model.signal_norm.parameters())
        + list(model.chunk_mha.parameters())
        + list(model.mha_norm.parameters())
        + list(model.bigru.parameters())
        + list(model.attn_layer.parameters())
        + list(model.classifier.parameters())
    )
    param_groups.append({
        "params"      : head_params,
        "lr"          : lr_head,
        "weight_decay": weight_decay,
        "name"        : "head",
    })

    param_groups = [g for g in param_groups if len(g["params"]) > 0]
    log.info(f"  LLRD param groups: {len(param_groups)}")
    for g in param_groups:
        n_total     = sum(p.numel() for p in g["params"])
        n_trainable = sum(p.numel() for p in g["params"] if p.requires_grad)
        log.info(f"    {g['name']:22s}  lr={g['lr']:.2e}  "
                 f"total={n_total:,}  trainable={n_trainable:,}")

    return AdamW(param_groups)


# ══════════════════════════════════════════════════════════════════════════════
# DEFERRED CLASS REWEIGHTING
# ══════════════════════════════════════════════════════════════════════════════
def compute_class_weights(labels_list, device):
    cnt   = Counter(labels_list)
    n     = len(labels_list)
    n_cls = len(cnt)
    w = torch.tensor(
        [n / (n_cls * cnt.get(i, 1)) for i in range(n_cls)],
        dtype=torch.float, device=device,
    ).clamp(0.5, 2.0)
    log.info(f"  Class weights (deferred) : {w.cpu().tolist()}")
    return w


# ══════════════════════════════════════════════════════════════════════════════
# TRAIN ONE EPOCH
# ══════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, scheduler, scaler,
                accum_steps, class_weights, epoch):
    model.train()
    total_loss = 0.0; n_correct = 0; n_total = 0
    optimizer.zero_grad()
    use_rw = (class_weights is not None) and (epoch >= DEFERRED_RW_EPOCH)

    pbar = tqdm(loader, desc=f"  Ep{epoch:02d} train", leave=False,
                dynamic_ncols=True)
    for step, batch in enumerate(pbar):
        ids  = batch["chunk_input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["chunk_attention_mask"].to(DEVICE, non_blocking=True)
        cmsk = batch["chunk_mask"].to(DEVICE, non_blocking=True)
        labs = batch["label"].to(DEVICE, non_blocking=True)
        sigs = batch["signal_id"].to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            out = model(ids, mask, cmsk, sigs, labs,
                        chunk_drop_prob=CHUNK_DROP_PROB)
            if use_rw:
                loss_raw = F.cross_entropy(
                    out.logits, labs,
                    weight=class_weights,
                    label_smoothing=LABEL_SMOOTHING,
                    reduction="mean",
                )
            else:
                loss_raw = out.loss
            loss = loss_raw / accum_steps

        scaler.scale(loss).backward()
        total_loss += loss_raw.item()
        preds       = torch.argmax(out.logits, dim=1)
        n_correct  += (preds == labs).sum().item()
        n_total    += len(labs)

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss_raw.item():.3f}",
                         acc=f"{n_correct/n_total:.3f}")

    return total_loss / len(loader), n_correct / n_total


# ══════════════════════════════════════════════════════════════════════════════
# EVALUATE
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(model, loader, class_weights=None, epoch=0):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0.0
    use_rw = (class_weights is not None) and (epoch >= DEFERRED_RW_EPOCH)

    with torch.no_grad():
        for batch in tqdm(loader, desc="  eval", leave=False,
                          dynamic_ncols=True):
            ids  = batch["chunk_input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["chunk_attention_mask"].to(DEVICE, non_blocking=True)
            cmsk = batch["chunk_mask"].to(DEVICE, non_blocking=True)
            labs = batch["label"].to(DEVICE, non_blocking=True)
            sigs = batch["signal_id"].to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):
                out = model(ids, mask, cmsk, sigs, labs, chunk_drop_prob=0.0)
                if use_rw:
                    loss_raw = F.cross_entropy(
                        out.logits, labs,
                        weight=class_weights,
                        label_smoothing=LABEL_SMOOTHING,
                    )
                else:
                    loss_raw = out.loss

            total_loss += loss_raw.item()
            probs = torch.softmax(out.logits.float(), dim=1).cpu().tolist()
            preds = torch.argmax(out.logits, dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labs.cpu().tolist())
            all_probs.extend([p[1] for p in probs])

    acc    = accuracy_score(all_labels, all_preds)
    f1     = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    prec   = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec    = recall_score(all_labels, all_preds, average="macro",    zero_division=0)
    f1_cls = f1_score(all_labels, all_preds, average=None,     zero_division=0)
    try:    auc = roc_auc_score(all_labels, all_probs)
    except: auc = 0.0
    try:    mcc = matthews_corrcoef(all_labels, all_preds)
    except: mcc = 0.0
    try:    kap = cohen_kappa_score(all_labels, all_preds)
    except: kap = 0.0

    dist = Counter(all_preds)
    if len(dist) < 2:
        log.warning(f"  ⚠️  Class collapse: {dict(dist)}")

    return {
        "loss"    : total_loss / len(loader),
        "acc"     : acc,  "f1"  : f1,
        "prec"    : prec, "rec" : rec,
        "auc"     : auc,  "mcc" : mcc, "kappa": kap,
        "f1_rej"  : float(f1_cls[0]) if len(f1_cls) > 0 else 0.0,
        "f1_acc"  : float(f1_cls[1]) if len(f1_cls) > 1 else 0.0,
        "preds"   : all_preds, "labels": all_labels, "probs": all_probs,
        "pred_dist": dict(dist),
    }


# ══════════════════════════════════════════════════════════════════════════════
# ADAPTIVE HYPERPARAMETER CONTROLLER
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveHPController:
    def __init__(self):
        self.overfit_streak  = 0
        self.dropout_bumped  = False
        self.wd_bumped       = False
        self.unfreeze_done   = False
        self.lr_bumped       = False

    def step(self, epoch, train_loss, val_loss, val_f1, model, optimizer):
        actions = []

        if val_loss - train_loss > OVERFIT_GAP_THRESH:
            self.overfit_streak += 1
        else:
            self.overfit_streak  = 0

        if self.overfit_streak >= OVERFIT_PATIENCE:
            if not self.dropout_bumped:
                for m in model.modules():
                    if isinstance(m, nn.Dropout):
                        m.p = min(m.p + 0.05, 0.4)
                self.dropout_bumped = True
                dp = [m.p for m in model.modules() if isinstance(m, nn.Dropout)]
                actions.append(f"dropout→{dp[0]:.2f}")
            elif not self.wd_bumped:
                for pg in optimizer.param_groups:
                    pg["weight_decay"] = min(pg["weight_decay"] * 1.5, 0.1)
                self.wd_bumped = True
                actions.append("weight_decay bumped")

        if epoch >= 8 and val_f1 < UNDERFIT_F1_THRESH:
            if not self.unfreeze_done:
                new_freeze = max(0, model.freeze_xlnet_layers - 3)
                model.unfreeze_xlnet_from(new_freeze)
                self.unfreeze_done = True
                actions.append(f"unfreeze XLNet layers ≥{new_freeze}")
            elif not self.lr_bumped:
                for pg in optimizer.param_groups:
                    if pg.get("name") == "head":
                        pg["lr"] = pg["lr"] * 1.2
                        actions.append(f"head_lr→{pg['lr']:.2e}")
                self.lr_bumped = True

        if actions:
            log.info(f"  [AdaptiveHP ep{epoch}] Actions: {' | '.join(actions)}")
        return actions


# ══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ══════════════════════════════════════════════════════════════════════════════
def save_plots(history, labels, preds, swa_start):
    ep         = [h["epoch"]      for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss   = [h["val_loss"]   for h in history]
    val_f1     = [h["val_f1"]     for h in history]
    val_acc    = [h["val_acc"]    for h in history]
    val_auc    = [h["val_auc"]    for h in history]
    val_mcc    = [h["val_mcc"]    for h in history]
    train_acc  = [h["train_acc"]  for h in history]
    f1_rej     = [h["val_f1_rej"] for h in history]
    f1_acc_cls = [h["val_f1_acc"] for h in history]

    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle("XLNet + SignalCrossAttn + MHA + BiGRU — 5000 Docs",
                 fontsize=14, fontweight="bold")

    ax = axes[0, 0]
    ax.plot(ep, train_loss, "b-o", ms=4, label="Train Loss")
    ax.plot(ep, val_loss,   "r-o", ms=4, label="Val Loss")
    if swa_start <= max(ep):
        ax.axvline(swa_start, color="orange", linestyle="--", alpha=0.7,
                   label=f"SWA start (ep{swa_start})")
    ax.fill_between(ep,
                    [abs(v - t) for v, t in zip(val_loss, train_loss)],
                    alpha=0.15, color="red", label="Overfit gap")
    ax.set_title("Loss Curve"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(ep, val_f1,    "g-s", ms=4, label="Val Macro-F1")
    ax.plot(ep, train_acc, "b-s", ms=4, label="Train Acc")
    ax.plot(ep, val_acc,   "r-s", ms=4, label="Val Acc")
    ax.axhline(SOTA_F1, color="purple", linestyle="--",
               label=f"SOTA F1={SOTA_F1}")
    ax.set_title("F1 / Accuracy"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)

    ax = axes[0, 2]
    ax.plot(ep, val_auc, "m-^", ms=4, label="Val AUC-ROC")
    ax.plot(ep, val_mcc, "c-^", ms=4, label="Val MCC")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_title("AUC & MCC"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(ep, f1_rej,     "r-o", ms=4, label="F1 REJECTED")
    ax.plot(ep, f1_acc_cls, "g-o", ms=4, label="F1 ACCEPTED")
    ax.set_title("Per-class F1"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)

    ax = axes[1, 1]
    gap = [v - t for v, t in zip(val_loss, train_loss)]
    ax.plot(ep, gap, "k-o", ms=4)
    ax.axhline(OVERFIT_GAP_THRESH, color="red", linestyle="--",
               label=f"Overfit thresh={OVERFIT_GAP_THRESH}")
    ax.axhline(0, color="gray", linestyle=":")
    ax.fill_between(ep, gap, 0,
                    where=[g > 0 for g in gap],
                    alpha=0.2, color="red",  label="Overfitting")
    ax.fill_between(ep, gap, 0,
                    where=[g <= 0 for g in gap],
                    alpha=0.2, color="blue", label="Underfitting")
    ax.set_title("Train-Val Loss Gap"); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 2]
    cm = confusion_matrix(labels, preds)
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["REJECTED", "ACCEPTED"])
    ax.set_yticklabels(["REJECTED", "ACCEPTED"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix — Best Epoch")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i][j]), ha="center", va="center",
                    fontsize=12, fontweight="bold",
                    color="white" if cm[i][j] > cm.max() / 2 else "black")

    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/full_analysis.png", dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"  Plots → {PLOT_DIR}/full_analysis.png")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":

    log.info("=" * 60 + "\n  LOADING DATA\n" + "=" * 60)
    records = load_data(INPUT_PATH)
    log.info(f"  QA pairs : {len(records):,}  |  "
             f"Docs : {len(set(r['doc_id'] for r in records)):,}")
    pool_records, pool_ids = build_balanced_pool(
        records, max_docs=MAX_TOTAL_DOCS, seed=SEED)
    train_records, val_records, n_tr, n_va = split_train_val(
        pool_records, pool_ids, seed=SEED)

    tokenizer = AutoTokenizer.from_pretrained(XLNET_MODEL_ID)
    if WITH_SIGNAL:
        tokenizer.add_tokens(["[FP]", "[FR]", "[N]"])
        log.info(f"  Vocab size : {len(tokenizer):,}")

    log.info("=" * 60 + "\n  PRE-TOKENISING\n" + "=" * 60)
    train_ds = HierarchicalLegalQADataset(
        train_records, tokenizer, MAX_CHUNK_LEN, MAX_CHUNKS, WITH_SIGNAL)
    val_ds   = HierarchicalLegalQADataset(
        val_records,   tokenizer, MAX_CHUNK_LEN, MAX_CHUNKS, WITH_SIGNAL)

    doc_labels_train = [s["label"].item() for s in train_ds.samples]
    cnt  = Counter(doc_labels_train)
    n0, n1 = cnt.get(0, 1), cnt.get(1, 1)
    log.info(f"  Train class dist → REJECTED={n0}  ACCEPTED={n1}")

    w = torch.tensor([
        len(doc_labels_train) / (2.0 * n0) if l == 0
        else len(doc_labels_train) / (2.0 * n1)
        for l in doc_labels_train
    ], dtype=torch.float)
    sampler       = WeightedRandomSampler(w, len(w), replacement=True)
    class_weights = compute_class_weights(doc_labels_train, DEVICE)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler,
        collate_fn=collate_fn, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
        collate_fn=collate_fn, num_workers=4, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
    )

    log.info("=" * 60 + "\n  BUILDING MODEL\n" + "=" * 60)
    model = HierarchicalXLNetBiGRU(
        model_id=XLNET_MODEL_ID, num_labels=2,
        dropout=DROPOUT,
        gru_hidden=GRU_HIDDEN,
        gru_layers=GRU_LAYERS,
        gru_dropout=GRU_DROPOUT,
        mha_heads=MHA_HEADS,
        mha_dropout=MHA_DROPOUT,
        label_smoothing=LABEL_SMOOTHING,
        freeze_xlnet_layers=FREEZE_XLNET_LAYERS,
    )
    if WITH_SIGNAL:
        model.resize_token_embeddings(len(tokenizer))
    model = model.to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    log.info(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    optimizer = build_llrd_optimizer(
        model, LR_XLNET, LR_HEAD, LLRD_DECAY, WEIGHT_DECAY)

    steps_per_epoch = (len(train_loader) + ACCUM_STEPS - 1) // ACCUM_STEPS
    total_steps     = steps_per_epoch * MAX_EPOCHS
    warmup_steps    = int(total_steps * WARMUP_RATIO)
    log.info(f"  Steps/ep={steps_per_epoch}  total={total_steps}  warmup={warmup_steps}")

    scheduler = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps)
    scaler    = GradScaler(enabled=USE_AMP)

    swa_model     = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR,
                          anneal_epochs=5, anneal_strategy="cos")
    swa_active    = False
    ahp           = AdaptiveHPController()

    start_epoch  = 1
    best_f1      = 0.0
    best_epoch   = 0
    best_metrics = {}
    no_improve   = 0
    history      = []

    if os.path.exists(CKPT_ROLL):
        try:
            ck = torch.load(CKPT_ROLL, map_location=DEVICE)
            model.load_state_dict(ck["model_state"])
            optimizer.load_state_dict(ck["optimizer_state"])
            scheduler.load_state_dict(ck["scheduler_state"])
            scaler.load_state_dict(ck["scaler_state"])
            start_epoch  = ck["epoch"] + 1
            best_f1      = ck["best_f1"]
            best_epoch   = ck["best_epoch"]
            best_metrics = ck["best_metrics"]
            no_improve   = ck["no_improve"]
            history      = ck["history"]
            if ck.get("swa_state"):
                swa_model.load_state_dict(ck["swa_state"])
            log.info(f"  ▶ RESUMED from epoch {ck['epoch']} "
                     f"(best F1={best_f1:.4f})")
        except Exception as e:
            log.warning(f"  ⚠️  Could not load checkpoint: {e} — starting fresh")

    csv_exists = os.path.exists(CSV_PATH) and start_epoch > 1
    csv_file   = open(CSV_PATH, "a" if csv_exists else "w", newline="")
    csv_writer = csv.writer(csv_file)
    if not csv_exists:
        csv_writer.writerow([
            "epoch","train_loss","train_acc",
            "val_loss","val_acc","val_f1","val_prec","val_rec",
            "val_auc","val_mcc","val_kappa",
            "val_f1_rej","val_f1_acc","overfit_gap",
            "swa_active","epoch_secs","adaptive_actions",
        ])

    log.info("=" * 60)
    log.info(f"  TRAINING — {MAX_EPOCHS} epochs | {n_tr} train | {n_va} val")
    log.info("=" * 60)

    start_time = datetime.now()

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        ep_start = datetime.now()

        if epoch >= SWA_START and not swa_active:
            swa_active = True
            log.info(f"  🔄  SWA activated at epoch {epoch}")

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, scaler,
            ACCUM_STEPS, class_weights, epoch)

        val_m = evaluate(model, val_loader, class_weights, epoch)

        if swa_active:
            swa_model.update_parameters(model)
            swa_scheduler.step()

        actions = ahp.step(
            epoch, train_loss, val_m["loss"], val_m["f1"], model, optimizer)

        ep_secs  = (datetime.now() - ep_start).total_seconds()
        done_min = (datetime.now() - start_time).total_seconds() / 60
        eta_min  = ep_secs * (MAX_EPOCHS - epoch) / 60
        gap      = val_m["loss"] - train_loss

        log.info(
            f"  Ep {epoch:02d}/{MAX_EPOCHS} | "
            f"TrLoss={train_loss:.4f} TrAcc={train_acc:.4f} | "
            f"VaLoss={val_m['loss']:.4f} VaAcc={val_m['acc']:.4f} "
            f"VaF1={val_m['f1']:.4f} | "
            f"AUC={val_m['auc']:.4f} MCC={val_m['mcc']:.4f} "
            f"κ={val_m['kappa']:.4f} | "
            f"F1[REJ={val_m['f1_rej']:.3f} ACC={val_m['f1_acc']:.3f}] | "
            f"Gap={gap:+.4f} SWA={'✓' if swa_active else '✗'} | "
            f"{ep_secs:.0f}s elapsed={done_min:.0f}m ETA≈{eta_min:.0f}m"
        )

        history.append({
            "epoch"     : epoch,
            "train_loss": round(train_loss,      4),
            "train_acc" : round(train_acc,       4),
            "val_loss"  : round(val_m["loss"],   4),
            "val_f1"    : round(val_m["f1"],     4),
            "val_acc"   : round(val_m["acc"],    4),
            "val_auc"   : round(val_m["auc"],    4),
            "val_mcc"   : round(val_m["mcc"],    4),
            "val_f1_rej": round(val_m["f1_rej"], 4),
            "val_f1_acc": round(val_m["f1_acc"], 4),
        })
        csv_writer.writerow([
            epoch, round(train_loss, 4), round(train_acc, 4),
            round(val_m["loss"],  4), round(val_m["acc"],   4),
            round(val_m["f1"],    4), round(val_m["prec"],  4),
            round(val_m["rec"],   4), round(val_m["auc"],   4),
            round(val_m["mcc"],   4), round(val_m["kappa"], 4),
            round(val_m["f1_rej"], 4), round(val_m["f1_acc"], 4),
            round(gap, 4), int(swa_active), round(ep_secs, 1),
            "|".join(actions),
        ])
        csv_file.flush()

        if val_m["f1"] > best_f1:
            best_f1 = val_m["f1"]; best_epoch = epoch
            best_metrics = val_m; no_improve = 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "best_f1": best_f1, "val_acc": val_m["acc"],
                "val_auc": val_m["auc"], "val_mcc": val_m["mcc"],
            }, CKPT_BEST)
            log.info(f"  ✅  New best F1={best_f1:.4f} → {CKPT_BEST}")
        else:
            no_improve += 1
            log.info(f"  No improve {no_improve}/{EARLY_STOP_PAT} "
                     f"(best F1={best_f1:.4f} @ ep {best_epoch})")

        torch.save({
            "epoch"          : epoch,
            "model_state"    : model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state"   : scaler.state_dict(),
            "swa_state"      : swa_model.state_dict() if swa_active else None,
            "best_f1"        : best_f1,
            "best_epoch"     : best_epoch,
            "best_metrics"   : best_metrics,
            "no_improve"     : no_improve,
            "history"        : history,
        }, CKPT_ROLL)

        if no_improve >= EARLY_STOP_PAT:
            log.info(f"  ⏹  Early stopping at epoch {epoch}")
            break

    csv_file.close()

    if swa_active:
        log.info("  🔄  Updating SWA BatchNorm statistics ...")
        update_bn(train_loader, swa_model, device=DEVICE)
        swa_val = evaluate(swa_model, val_loader, class_weights, MAX_EPOCHS)
        log.info(f"  SWA model → F1={swa_val['f1']:.4f}  "
                 f"Acc={swa_val['acc']:.4f}  AUC={swa_val['auc']:.4f}")
        if swa_val["f1"] > best_f1:
            torch.save({"model_state": swa_model.state_dict(),
                        "source": "SWA", "f1": swa_val["f1"]},
                       f"{OUTPUT_DIR}/swa_best_model.pt")
            log.info("  ✅  SWA model is best → saved")

    total_mins = (datetime.now() - start_time).total_seconds() / 60
    report = classification_report(
        best_metrics["labels"], best_metrics["preds"],
        target_names=["REJECTED", "ACCEPTED"], digits=4,
    )
    log.info("\n" + "=" * 60)
    log.info(f"  FINAL RESULTS  (best epoch = {best_epoch})")
    log.info("=" * 60)
    log.info(f"  Val Acc   : {best_metrics['acc']:.4f}   SOTA={SOTA_ACC}")
    log.info(f"  Val F1    : {best_metrics['f1']:.4f}   SOTA={SOTA_F1}")
    log.info(f"  Val AUC   : {best_metrics['auc']:.4f}")
    log.info(f"  Val MCC   : {best_metrics['mcc']:.4f}")
    log.info(f"  Val κ     : {best_metrics['kappa']:.4f}")
    log.info(f"  F1 REJ    : {best_metrics['f1_rej']:.4f}")
    log.info(f"  F1 ACC    : {best_metrics['f1_acc']:.4f}")
    log.info(f"  Runtime   : {total_mins:.1f} min")
    log.info(f"\n{report}")

    save_plots(history, best_metrics["labels"],
               best_metrics["preds"], SWA_START)

    log.info(f"  Best model  → {CKPT_BEST}")
    log.info(f"  Last ckpt   → {CKPT_ROLL}  (resume-safe)")
    log.info(f"  CSV         → {CSV_PATH}")
    log.info(f"  Plots       → {PLOT_DIR}/full_analysis.png")
    log.info(f"  Log         → {log_file}")
    log.info("  ✅  Done.")

09:31:36 | Device        : cuda  |  AMP: True
09:31:36 | Architecture  : XLNet → SignalCrossAttn → MHA(8h) → BiGRU → AttnPool → Linear
09:31:36 | Epochs        : 50  patience=15  SWA from ep 35
09:31:36 | LR XLNET/HEAD : 2e-05/1e-05  LLRD=0.95  WD=0.01
09:31:36 | Novel         : SignalCrossAttn + DeferredRW(ep6) + LLRD + ChunkDrop + SWA + AdaptiveHP
09:31:36 | Fix           : All XLNet layers pre-registered in optimizer — no runtime add_param_group
09:31:37 | ============================================================
  LOADING DATA
09:31:37 |   QA pairs : 45,000  |  Docs : 6,680
09:31:37 |   Balanced pool : 5,000 docs  (2500 per class)  QA=33,590
09:31:37 |   Train : 4000 docs (26,940 QA)  |  Val : 1000 docs (6,650 QA)
09:31:39 |   Vocab size : 32,003
09:31:39 | ============================================================
  PRE-TOKENISING


  tokenising:   0%|          | 0/4000 [00:00<?, ?it/s]

09:31:46 |   Dataset ready : 4000 docs (pre-tokenised)


  tokenising:   0%|          | 0/1000 [00:00<?, ?it/s]

09:31:47 |   Dataset ready : 1000 docs (pre-tokenised)
09:31:47 |   Train class dist → REJECTED=2223  ACCEPTED=1777
09:31:48 |   Class weights (deferred) : [0.8996850848197937, 1.1254924535751343]
09:31:48 | ============================================================
  BUILDING MODEL
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
09:31:50 |   Trainable: 53,561,859 / 124,210,947 (43.1%)
09:31:50 |   LLRD param groups: 15
09:31:50 |     xlnet_word_emb          lr=1.08e-05  total=24,578,304  trainable=0
09:31:50 |     xlnet_mask_emb          lr=1.08e-05  total=768  trainable=768
09:31:50 |     xlnet_layer_00          lr=1.08e-05  total=7,678,464  trainable=0
09:31:50 |     xlnet_layer_01          lr=1.14e-05  total=7,678,464  trainable=0
09:31:50 |     xlnet_layer_02          lr=1.20e

...

  eval:   0%|                                                                                    | 0/63 [00:00…

10:59:05 |   Ep 31/50 | TrLoss=0.4287 TrAcc=0.8195 | VaLoss=0.8134 VaAcc=0.6510 VaF1=0.6283 | AUC=0.7475 MCC=0.3139 κ=0.2867 | F1[REJ=0.537 ACC=0.720] | Gap=+0.3847 SWA=✗ | 95s elapsed=87m ETA≈30m
10:59:05 |   No improve 14/15 (best F1=0.6726 @ ep 17)


  Ep32 train:   0%|                                                                             | 0/500 [00:00…

  eval:   0%|                                                                                    | 0/63 [00:00…

11:00:41 |   Ep 32/50 | TrLoss=0.4278 TrAcc=0.8215 | VaLoss=0.7198 VaAcc=0.6900 VaF1=0.6484 | AUC=0.7400 MCC=0.3133 κ=0.3050 | F1[REJ=0.527 ACC=0.769] | Gap=+0.2920 SWA=✗ | 95s elapsed=89m ETA≈29m
11:00:41 |   No improve 15/15 (best F1=0.6726 @ ep 17)
11:00:43 |   ⏹  Early stopping at epoch 32
11:00:43 | 
11:00:43 |   FINAL RESULTS  (best epoch = 17)
11:00:43 | ============================================================
11:00:43 |   Val Acc   : 0.7410   SOTA=0.78
11:00:43 |   Val F1    : 0.6726   SOTA=0.8131
11:00:43 |   Val AUC   : 0.7579
11:00:43 |   Val MCC   : 0.3455
11:00:43 |   Val κ     : 0.3454
11:00:43 |   F1 REJ    : 0.5230
11:00:43 |   F1 ACC    : 0.8222
11:00:43 |   Runtime   : 88.9 min
11:00:43 | 
              precision    recall  f1-score   support

    REJECTED     0.5338    0.5126    0.5230       277
    ACCEPTED     0.8161    0.8285    0.8222       723

    accuracy                         0.7410      1000
   macro avg     0.6750    0.6706    0.6726      1000
weighte